
# Binary Classification From Scratch — ML Foundations

> **Interactive learning notebook:** change the dataset, features, model, hyperparameters, threshold and training settings, then rerun the experiment to see how the mathematics changes the model.

**No scikit-learn is used.** Everything important is implemented with NumPy, Matplotlib and ipywidgets.

### Concepts covered

- Synthetic binary datasets and controllable sample/feature count
- Linear model: \(z=Xw+b\)
- Sigmoid probability
- Binary cross-entropy
- Gradient derivation and gradient descent
- Batch and mini-batch training
- Train / validation / test split
- Logistic regression / single neuron
- Decision boundary
- Threshold selection
- Confusion matrix, accuracy, precision, recall, F1
- TPR, FPR, ROC and AUC
- XOR and the limitation of linear models
- Hidden layers and nonlinear activations
- Forward propagation
- Backpropagation
- ReLU vs sigmoid hidden activations
- Multilayer perceptron (MLP)
- L2 regularization
- Overfitting experiments
- Weight visualization



## 1. The mathematical pipeline

For a binary classifier:

\[
z = Xw+b
\]

\[
p=\sigma(z)=\frac{1}{1+e^{-z}}
\]

\[
L=-\frac1N\sum_i\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right]
\]

For logistic regression with BCE:

\[
\frac{\partial L}{\partial z}=p-y
\]

\[
\frac{\partial L}{\partial w}=\frac1N X^T(p-y),\qquad
\frac{\partial L}{\partial b}=\frac1N\sum_i(p_i-y_i)
\]

and gradient descent updates:

\[
w\leftarrow w-\eta\frac{\partial L}{\partial w},\qquad
b\leftarrow b-\eta\frac{\partial L}{\partial b}
\]

The interactive experiment below lets you change \(\eta\), epochs, batch size, regularization, threshold, dataset size and feature count and observe the result.


In [ ]:

# Setup
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

np.set_printoptions(precision=4, suppress=True)

def sigmoid(z):
    z = np.clip(z, -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

def bce(y, p):
    eps = 1e-12
    p = np.clip(p, eps, 1-eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

def accuracy(y, pred):
    return np.mean(y == pred)

def precision(y, pred):
    tp = np.sum((y==1)&(pred==1))
    fp = np.sum((y==0)&(pred==1))
    return tp/(tp+fp) if tp+fp else 0.0

def recall(y, pred):
    tp = np.sum((y==1)&(pred==1))
    fn = np.sum((y==1)&(pred==0))
    return tp/(tp+fn) if tp+fn else 0.0

def f1_score(y, pred):
    p, r = precision(y, pred), recall(y, pred)
    return 2*p*r/(p+r) if p+r else 0.0

def confusion_counts(y, pred):
    tn = np.sum((y==0)&(pred==0))
    fp = np.sum((y==0)&(pred==1))
    fn = np.sum((y==1)&(pred==0))
    tp = np.sum((y==1)&(pred==1))
    return tn, fp, fn, tp

def roc_curve_manual(y, scores):
    thresholds = np.r_[np.inf, np.sort(np.unique(scores))[::-1], -np.inf]
    tpr, fpr = [], []
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tn, fp, fn, tp = confusion_counts(y, pred)
        tpr.append(tp/(tp+fn) if tp+fn else 0.0)
        fpr.append(fp/(fp+tn) if fp+tn else 0.0)
    order = np.argsort(fpr)
    fpr, tpr = np.array(fpr)[order], np.array(tpr)[order]
    auc = np.trapezoid(tpr, fpr)
    return fpr, tpr, auc

def split_data(X, y, train_frac=0.60, val_frac=0.20, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n_train = int(len(X)*train_frac)
    n_val = int(len(X)*val_frac)
    tr, va, te = idx[:n_train], idx[n_train:n_train+n_val], idx[n_train+n_val:]
    return X[tr], y[tr], X[va], y[va], X[te], y[te]

def standardize_fit(X_train, X_val, X_test):
    mu = X_train.mean(axis=0, keepdims=True)
    sd = X_train.std(axis=0, keepdims=True)
    sd = np.where(sd < 1e-8, 1.0, sd)
    return (X_train-mu)/sd, (X_val-mu)/sd, (X_test-mu)/sd, mu, sd

print("Ready.")


## 2. Interactive controls
Change any control and click **Run experiment**. The notebook generates a fresh dataset and trains both a linear classifier and an MLP.

In [ ]:

# Interactive controls
dataset = widgets.Dropdown(
    options=["Linear", "Nonlinear circles", "XOR"],
    value="Linear",
    description="Dataset:"
)
n_samples = widgets.IntSlider(value=240, min=40, max=1000, step=20, description="Samples:")
n_features = widgets.IntSlider(value=2, min=2, max=8, step=1, description="Features:")
noise = widgets.FloatSlider(value=0.25, min=0.0, max=1.5, step=0.05, description="Noise:")
seed = widgets.IntSlider(value=42, min=0, max=999, step=1, description="Seed:")

lr = widgets.FloatLogSlider(value=0.05, base=10, min=-3, max=0, step=0.05, description="Learning rate:")
epochs = widgets.IntSlider(value=500, min=20, max=3000, step=20, description="Epochs:")
batch_size = widgets.IntSlider(value=32, min=1, max=256, step=1, description="Batch size:")
threshold = widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05, description="Threshold:")
l2 = widgets.FloatLogSlider(value=0.001, base=10, min=-5, max=0, step=0.25, description="L2 λ:")

hidden_units = widgets.IntSlider(value=8, min=1, max=32, step=1, description="Hidden units:")
activation = widgets.Dropdown(options=["ReLU", "Sigmoid"], value="ReLU", description="Hidden act:")
mlp_lr = widgets.FloatLogSlider(value=0.1, base=10, min=-3, max=0, step=0.05, description="MLP LR:")
mlp_epochs = widgets.IntSlider(value=1000, min=50, max=5000, step=50, description="MLP epochs:")

run_button = widgets.Button(description="▶ Run experiment", button_style="success")
output = widgets.Output()

controls = widgets.VBox([
    widgets.HTML("<b>Dataset</b>"),
    widgets.HBox([dataset, n_samples, n_features]),
    widgets.HBox([noise, seed]),
    widgets.HTML("<b>Logistic regression / single neuron</b>"),
    widgets.HBox([lr, epochs, batch_size]),
    widgets.HBox([threshold, l2]),
    widgets.HTML("<b>MLP / hidden-layer experiment</b>"),
    widgets.HBox([hidden_units, activation]),
    widgets.HBox([mlp_lr, mlp_epochs]),
    run_button
])
display(controls)


In [ ]:

# Dataset generation

def make_dataset(kind, n, d, noise_level, seed_value):
    rng = np.random.default_rng(seed_value)

    if kind == "Linear":
        d = max(2, d)
        X = rng.normal(size=(n, d))
        true_w = rng.normal(size=d)
        true_w /= np.linalg.norm(true_w) + 1e-12
        score = X @ true_w + noise_level*rng.normal(size=n)
        y = (score > 0).astype(int)

    elif kind == "Nonlinear circles":
        d = max(2, d)
        angles = rng.uniform(0, 2*np.pi, n)
        radii = np.where(np.arange(n) % 2 == 0,
                         1.0 + noise_level*rng.normal(size=n),
                         2.5 + noise_level*rng.normal(size=n))
        X2 = np.c_[radii*np.cos(angles), radii*np.sin(angles)]
        if d > 2:
            extra = rng.normal(scale=0.5, size=(n, d-2))
            X = np.c_[X2, extra]
        else:
            X = X2
        y = (radii > np.median(radii)).astype(int)

    else:  # XOR
        d = max(2, d)
        base = rng.integers(0, 2, size=(n, 2))
        X2 = base + noise_level*rng.normal(size=(n, 2))
        y = (base[:,0] ^ base[:,1]).astype(int)
        if d > 2:
            extra = rng.normal(scale=0.5, size=(n, d-2))
            X = np.c_[X2, extra]
        else:
            X = X2

    return X.astype(float), y.astype(int)

def plot_dataset(X, y, title):
    plt.figure(figsize=(6,5))
    plt.scatter(X[:,0], X[:,1], c=y, cmap="coolwarm", edgecolor="k", alpha=0.75)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.show()


In [ ]:

# Logistic regression from scratch

def train_logistic(X, y, learning_rate, epochs, batch_size, l2_lambda, seed_value):
    rng = np.random.default_rng(seed_value)
    n, d = X.shape
    w = rng.normal(scale=0.01, size=d)
    b = 0.0
    history = []

    for epoch in range(epochs):
        order = rng.permutation(n)
        Xs, ys = X[order], y[order]

        for start in range(0, n, batch_size):
            xb = Xs[start:start+batch_size]
            yb = ys[start:start+batch_size]

            p = sigmoid(xb @ w + b)
            error = p - yb

            dw = (xb.T @ error) / len(xb) + l2_lambda*w
            db = np.mean(error)

            w -= learning_rate*dw
            b -= learning_rate*db

        p_all = sigmoid(X @ w + b)
        loss = bce(y, p_all) + 0.5*l2_lambda*np.sum(w*w)
        history.append(loss)

    return w, b, np.array(history)

def decision_boundary_2d(w, b, xlim=(-3,3)):
    x1 = np.linspace(xlim[0], xlim[1], 200)
    if abs(w[1]) < 1e-10:
        return x1, np.full_like(x1, np.nan)
    return x1, -(w[0]*x1+b)/w[1]


In [ ]:

# MLP from scratch: input -> hidden -> output

def train_mlp(X, y, hidden, hidden_activation, learning_rate, epochs, l2_lambda, seed_value):
    rng = np.random.default_rng(seed_value)
    n, d = X.shape

    scale1 = np.sqrt(2/d) if hidden_activation == "ReLU" else np.sqrt(1/d)
    W1 = rng.normal(scale=scale1, size=(d, hidden))
    b1 = np.zeros((1, hidden))
    W2 = rng.normal(scale=np.sqrt(2/hidden), size=(hidden, 1))
    b2 = np.zeros((1,1))
    history = []

    for epoch in range(epochs):
        z1 = X @ W1 + b1
        if hidden_activation == "ReLU":
            a1 = relu(z1)
            da1 = relu_grad(z1)
        else:
            a1 = sigmoid(z1)
            da1 = a1*(1-a1)

        z2 = a1 @ W2 + b2
        yhat = sigmoid(z2)

        loss = bce(y.reshape(-1,1), yhat) + 0.5*l2_lambda*(np.sum(W1*W1)+np.sum(W2*W2))
        history.append(loss)

        dz2 = yhat - y.reshape(-1,1)
        dW2 = (a1.T @ dz2)/n + l2_lambda*W2
        db2 = np.mean(dz2, axis=0, keepdims=True)

        dz1 = (dz2 @ W2.T) * da1
        dW1 = (X.T @ dz1)/n + l2_lambda*W1
        db1 = np.mean(dz1, axis=0, keepdims=True)

        W2 -= learning_rate*dW2
        b2 -= learning_rate*db2
        W1 -= learning_rate*dW1
        b1 -= learning_rate*db1

    return W1, b1, W2, b2, np.array(history)

def mlp_predict(X, params, hidden_activation):
    W1,b1,W2,b2 = params
    z1 = X @ W1 + b1
    a1 = relu(z1) if hidden_activation == "ReLU" else sigmoid(z1)
    return sigmoid(a1 @ W2 + b2).ravel()


In [ ]:

# One-button experiment runner

def run_experiment(_=None):
    with output:
        clear_output(wait=True)

        X, y = make_dataset(dataset.value, n_samples.value, n_features.value,
                            noise.value, seed.value)
        Xtr, ytr, Xva, yva, Xte, yte = split_data(X, y, seed=seed.value)
        Xtr, Xva, Xte, mu, sd = standardize_fit(Xtr, Xva, Xte)

        print(f"Dataset: {dataset.value} | samples={len(X)} | features={X.shape[1]}")
        print(f"Train/validation/test: {len(Xtr)}/{len(Xva)}/{len(Xte)}")
        plot_dataset(X, y, f"{dataset.value}: generated data (first 2 features)")

        # Logistic model
        w,b,loss_log = train_logistic(
            Xtr, ytr, lr.value, epochs.value, batch_size.value, l2.value, seed.value
        )
        p_tr = sigmoid(Xtr@w+b)
        p_va = sigmoid(Xva@w+b)
        p_te = sigmoid(Xte@w+b)

        pred_te = (p_te >= threshold.value).astype(int)
        tn,fp,fn,tp = confusion_counts(yte,pred_te)
        fpr,tpr,auc = roc_curve_manual(yte,p_te)

        # MLP
        mlp_params = train_mlp(
            Xtr, ytr, hidden_units.value, activation.value,
            mlp_lr.value, mlp_epochs.value, l2.value, seed.value
        )
        p_mlp = mlp_predict(Xte, mlp_params, activation.value)
        pred_mlp = (p_mlp >= threshold.value).astype(int)

        print("\n=== Logistic regression / single neuron ===")
        print(f"Accuracy : {accuracy(yte,pred_te):.4f}")
        print(f"Precision: {precision(yte,pred_te):.4f}")
        print(f"Recall   : {recall(yte,pred_te):.4f}")
        print(f"F1       : {f1_score(yte,pred_te):.4f}")
        print(f"ROC-AUC  : {auc:.4f}")

        print("\n=== MLP ===")
        print(f"Hidden layer: {hidden_units.value} units, {activation.value}")
        print(f"Accuracy : {accuracy(yte,pred_mlp):.4f}")
        print(f"Precision: {precision(yte,pred_mlp):.4f}")
        print(f"Recall   : {recall(yte,pred_mlp):.4f}")
        print(f"F1       : {f1_score(yte,pred_mlp):.4f}")

        # Loss curves
        plt.figure(figsize=(7,4))
        plt.plot(loss_log, label="Logistic / single neuron")
        plt.plot(mlp_params[-1], label="MLP")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Learning curves")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.show()

        # Decision boundary for 2D
        if X.shape[1] == 2:
            plt.figure(figsize=(6,5))
            plt.scatter(Xtr[:,0], Xtr[:,1], c=ytr, cmap="coolwarm", alpha=0.65, edgecolor="k")
            xx = np.linspace(Xtr[:,0].min()-1, Xtr[:,0].max()+1, 300)
            if abs(w[1]) > 1e-10:
                yy = -(w[0]*xx+b)/w[1]
                plt.plot(xx,yy,linewidth=2,label="Logistic boundary")
            plt.xlabel("Feature 1 (standardized)")
            plt.ylabel("Feature 2 (standardized)")
            plt.title("Linear decision boundary")
            plt.legend()
            plt.grid(alpha=0.25)
            plt.show()

        # Confusion matrix
        cm = np.array([[tn,fp],[fn,tp]])
        plt.figure(figsize=(5,4))
        plt.imshow(cm, cmap="Blues")
        plt.title(f"Confusion matrix — threshold={threshold.value:.2f}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        for i in range(2):
            for j in range(2):
                plt.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=14)
        plt.xticks([0,1],["0","1"])
        plt.yticks([0,1],["0","1"])
        plt.colorbar()
        plt.show()

        # ROC
        plt.figure(figsize=(6,5))
        plt.plot(fpr,tpr,label=f"AUC = {auc:.3f}")
        plt.plot([0,1],[0,1],"--",label="Random")
        plt.xlabel("False positive rate")
        plt.ylabel("True positive rate")
        plt.title("ROC curve")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.show()

        # Probability distribution
        plt.figure(figsize=(7,4))
        plt.hist(p_te[yte==0], bins=15, alpha=0.65, label="Class 0")
        plt.hist(p_te[yte==1], bins=15, alpha=0.65, label="Class 1")
        plt.axvline(threshold.value, linestyle="--", label="Threshold")
        plt.xlabel("Predicted probability of class 1")
        plt.ylabel("Count")
        plt.title("Predicted probability distributions")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.show()

        # MLP hidden-unit activations / weights
        W1,b1,W2,b2,_ = mlp_params
        plt.figure(figsize=(7,4))
        plt.bar(np.arange(len(W2.ravel())), W2.ravel())
        plt.xlabel("Hidden neuron")
        plt.ylabel("Output weight")
        plt.title("MLP hidden → output weights")
        plt.grid(alpha=0.25)
        plt.show()

        print("\nInterpretation:")
        if dataset.value == "XOR" or dataset.value == "Nonlinear circles":
            print("The dataset is nonlinear. A single linear decision boundary is fundamentally limited; the hidden layer gives the MLP extra representational power.")
        else:
            print("The dataset is approximately linearly separable. Logistic regression may already perform strongly.")
        print("Try changing one parameter at a time and rerun the experiment to connect the observed plots to the underlying equations.")

run_button.on_click(run_experiment)
display(output)



## 3. Threshold experiment

The model outputs a **probability**, not a hard class. The usual rule is

\[
\hat y =
\begin{cases}
1 & p\ge t\\
0 & p<t
\end{cases}
\]

Change the threshold above and rerun. A lower threshold usually increases recall while potentially increasing false positives; a higher threshold usually does the opposite.

**Experiment:** try \(t=0.2,\ 0.5,\ 0.8\) and compare the confusion matrix, precision and recall.



## 4. Why XOR breaks a single neuron

For XOR:

| \(x_1\) | \(x_2\) | \(y\) |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

No single straight line can separate the two classes.

Run the notebook with **Dataset = XOR**, then compare:

- logistic regression / single neuron
- MLP with a hidden layer
- loss curves
- learned output weights

The key idea is:

\[
\text{linear model} \rightarrow \text{linear decision boundary}
\]

but

\[
\text{linear}+\text{nonlinear hidden layer} \rightarrow \text{nonlinear decision boundary}
\]



## 5. Backpropagation — what the MLP is actually computing

For one hidden layer:

\[
Z_1=XW_1+b_1
\]

\[
A_1=f(Z_1)
\]

\[
Z_2=A_1W_2+b_2
\]

\[
\hat y=\sigma(Z_2)
\]

For BCE + sigmoid output:

\[
\delta_2=\hat y-y
\]

\[
dW_2=A_1^T\delta_2,\qquad db_2=\text{mean}(\delta_2)
\]

and

\[
\delta_1=(\delta_2W_2^T)\odot f'(Z_1)
\]

\[
dW_1=X^T\delta_1,\qquad db_1=\text{mean}(\delta_1)
\]

The code above implements these equations directly. There is no automatic differentiation and no ML framework hiding the mathematics.



## 6. Mini-batch gradient descent

If `batch_size = number of training samples`, the update is batch gradient descent.

If `batch_size` is smaller, each epoch contains several updates:

\[
\theta\leftarrow\theta-\eta\nabla_\theta L_{\text{mini-batch}}
\]

**Experiment:** keep the dataset and learning rate fixed, then compare `batch_size = 1`, `32`, and the full training set.

Watch the learning curve and training speed.



## 7. Overfitting and L2 regularization

L2 regularization adds:

\[
L_{\text{total}}=L+\frac{\lambda}{2}
\left(\|W_1\|_F^2+\|W_2\|_F^2\right)
\]

so the weight gradients become:

\[
dW\leftarrow dW+\lambda W
\]

Try:

- many hidden units
- high noise
- fewer training samples
- `L2 λ = 0`
- then increase `L2 λ`

The goal is not simply to minimize training loss; the real objective is **generalization to unseen data**.



## 8. Suggested experiments

### Experiment A — Learn logistic regression
- Dataset: Linear
- Features: 2
- Batch size: 32
- Observe the linear boundary and loss curve.

### Experiment B — Break the linear model
- Dataset: XOR
- Compare logistic regression and MLP.
- Change hidden units from 1 → 2 → 8.

### Experiment C — Activation functions
- Dataset: XOR or Nonlinear circles
- Compare ReLU and sigmoid hidden layers.

### Experiment D — Learning rate
Try:
- 0.001
- 0.01
- 0.1
- 1.0

Observe slow learning, stable learning and unstable/overshooting behavior.

### Experiment E — Threshold
Try:
- 0.2
- 0.5
- 0.8

Compare precision, recall and the confusion matrix.

### Experiment F — Regularization
Increase hidden units and noise, then compare:
- \(\lambda=0\)
- \(\lambda=0.001\)
- \(\lambda=0.01\)
- \(\lambda=0.1\)

### Experiment G — Feature dimensionality
Increase `Features` from 2 to 8. The training model still works, but the 2-D decision-boundary visualization is intentionally unavailable because the real decision surface now lives in a higher-dimensional space.



## 9. Final mental model

```text
DATA
  ↓
features X + labels y
  ↓
MODEL
  ├── Logistic regression / single neuron
  │      z = Xw + b
  │      p = sigmoid(z)
  │
  └── MLP
         X → W1,b1 → activation → W2,b2 → sigmoid
  ↓
LOSS
  BCE + optional L2
  ↓
BACKPROPAGATION
  gradients via chain rule
  ↓
OPTIMIZER
  gradient descent / mini-batches
  ↓
PROBABILITIES
  ↓
THRESHOLD
  ↓
PREDICTIONS
  ↓
EVALUATION
  ├── Accuracy
  ├── Precision
  ├── Recall
  ├── F1
  ├── Confusion matrix
  └── ROC / AUC
```

**Next stage:** multiclass classification and Softmax.
